In [1]:
import polars as pl
from soupsieve import select
import os
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
from gensim.models import Word2Vec
import numpy as np
from cuml.neighbors import NearestNeighbors

In [2]:
articles_path='../data/articles.parquet'
customer_path='../data/customers.parquet'
transaction_path='../data/transactions.parquet'

In [3]:
articles=pl.read_parquet(articles_path)
transactions=pl.read_parquet(transaction_path)

In [4]:
sentences=(
    transactions.sort(['customer_id','time'])
    .group_by('customer_id')
    .agg(pl.col('article_id').tail(30))
    ['article_id']
    .to_list()
)

In [5]:
# model=Word2Vec(
#     sentences=sentences,
#     vector_size=64,
#     window=5,
#     min_count=5,
#     workers=5,
#     sg=1,
#     epochs=5,
# )

In [6]:
# model.save('../save/model/item2vec.model')

In [7]:
model=Word2Vec.load('../save/model/item2vec.model')

In [8]:
idx2aid=model.wv.index_to_key

In [9]:
embeddings = np.array([model.wv[aid] for aid in model.wv.index_to_key], dtype=np.float32)# 词的向量
d=model.wv.vectors.shape[1]

In [10]:
knn=NearestNeighbors(n_neighbors=51,metric='euclidean') # 最近邻查找，每个样本返回21个（包括自己）
knn.fit(embeddings)

NearestNeighbors()

In [11]:
_,aid_nns=knn.kneighbors(embeddings)

In [12]:
aid_nns=aid_nns[:,1:]
aid_nns

array([[   40,     1,    10, ..., 64003, 71343, 76024],
       [   40,    31,     0, ..., 69363, 64465, 80947],
       [    5,   130,    21, ..., 65808, 81112,   208],
       ...,
       [81645, 83369, 81242, ..., 82488, 82605, 81562],
       [81908, 81422, 82023, ..., 75440, 82051, 79536],
       [81158, 82683, 83141, ..., 82452, 83294, 80716]])

In [13]:
item2item={idx2aid[aid]:[idx2aid[i] for i in row] for aid,row in enumerate(aid_nns)}

In [14]:
def train_w2vec(data,is_valid=False,model_path='../save/model/item2vec.model'):
    sentences=(
        data.sort(['customer_id','time'])
        .group_by('customer_id', maintain_order=True)
        .agg(pl.col('article_id').tail(30))
        ['article_id']
        .to_list()
    )
    if os.path.exists(model_path) and not is_valid:model=Word2Vec.load(model_path)
    else:
        model=Word2Vec(
            sentences=sentences,
            vector_size=64,
            window=5,
            min_count=5,
            workers=5,
            sg=1,
            epochs=5,
        )
    if not is_valid:model.save('../save/model/item2vec.model')

    return model

In [15]:
from sklearn.metrics.pairwise import cosine_similarity
def recall_item(model,topk):
    embeddings = np.array([model.wv[aid] for aid in model.wv.index_to_key], dtype=np.float32)# 词的向量
    knn=NearestNeighbors(n_neighbors=topk+1,metric='cosine') # 最近邻查找，每个样本返回21个（包括自己）
    knn.fit(embeddings)

    _,aid_nns=knn.kneighbors(embeddings)
    aid_nns=aid_nns[:,1:]

    idx2aid=model.wv.index_to_key

    item2item={}

    for idx, row in enumerate(aid_nns):
        src_vec = embeddings[idx].reshape(1, -1)
        nbr_vecs = embeddings[row]

        cos = cosine_similarity(src_vec, nbr_vecs)[0] # 返回[1,k]的矩阵[[cos_1,cos_2,...,cos_k]]

        item2item[idx2aid[idx]] = [
            (idx2aid[i], float(c))
            for i, c in zip(row, cos)
        ]
    return item2item

In [16]:
def recall_w2vec(data,topk=50,hist_len=10,is_valid=False,model_path='../save/model/item2vec.model'):
    rows=[]
    model=train_w2vec(data,is_valid,model_path)
    item2item=recall_item(model,topk)

    customer_ids=[]
    article_aids=[]
    scores=[]

    for cid,g in tqdm(data.group_by('customer_id'),total=data['customer_id'].unique().shape[0]):
        hist=(
            g.sort('time',descending=True)
            .select('article_id')
            .head(hist_len)['article_id']
            .to_list()
        )
        score = defaultdict(float)

        for i, aid in enumerate(hist):
            if aid not in item2item:
                continue
            w = 1.0 / (i + 1)

            for j,(rec_aid, cos) in enumerate(item2item[aid]):
                if rec_aid in hist:
                    continue
                score[rec_aid] += w/(j+1) # 越相似的物品权重越大

        res=sorted(score.items(),key=lambda x:x[1],reverse=True)[:topk]



        for aid, score in res:
            customer_ids.append(cid[0])
            article_aids.append(aid)
            scores.append(score)   # 这里已不再是 rank
    df = pl.DataFrame({
        "customer_id": customer_ids,
        "article_id": article_aids,
        "score": scores,
    })

    return df

In [17]:
def get_validation_data(data: pl.DataFrame):
    DAY = 86400
    WEEK = 7 * DAY

    max_time = data.select(pl.col("time").max()).item()

    valid_start = max_time - 6 * DAY
    train_start = valid_start - 6 * WEEK

    train_df = data.filter(
        (pl.col("time") >= train_start) &
        (pl.col("time") <  valid_start)
    )

    valid_df = data.filter(
        pl.col("time") >= valid_start
    )

    return train_df, valid_df

In [18]:
def metric_recall(data,topk=5):
    train_df,valid_df=get_validation_data(data)

    user_item=recall_w2vec(train_df,topk*10,5,True)
    user_set=set(user_item['customer_id'])

    pred_df = (
        user_item
        .group_by("customer_id")
        .agg(pl.col("article_id").alias("pred_items"))
    )

    true_df = (
        valid_df
    .group_by("customer_id")
        .agg(pl.col('article_id').unique().alias("true_items")))

    eval_df=pred_df.join(true_df,on='customer_id',how='inner')

    for k in range(10, topk * 10 + 1, 10):
        total = 0.0
        cnt = 0

        for pred, true in zip(eval_df["pred_items"], eval_df["true_items"]):
            if len(true) == 0:
                continue
            total += len(set(pred[:k]) & set(true)) / len(true)
            cnt += 1

        print(f"Recall@{k}: {total / cnt:.6f}")

In [19]:
# metric_recall(transactions)

In [20]:
res=recall_w2vec(transactions)

100%|██████████| 1362281/1362281 [06:46<00:00, 3352.31it/s]


In [21]:
res.write_parquet('../save/candidate/w2vec.parquet')